# WP3 Africa Hazard Ranking — Dataset Notebook 03  
## INFORM Climate Change (Brochure data) — pessimistic scenario extraction (2050)

**Goal (dataset-specific):**  
Extract WP3-relevant indicators from **INFORM Climate Change Brochure data** for the **pessimistic scenario** and **2050**, then upsert into:

`data/intermediate/wp3_country_indicators_master.csv`

**Key constraint:**  
INFORM CC brochure data **does not contain ISO3 codes**.  
We therefore build an internal ISO3 mapping using **INFORM Risk (Mid 2025)** which *does* contain ISO3 and country names, then perform robust name matching.

---
## Scenario choice
Use **PESSIMISTIC** scenario only:
- **RCP 8.5** (high emissions) + **SSP 3** (Regional Rivalry)  
(We store this explicitly in `notes` and in `indicator_id` suffix.)

---
## Indicators extracted

**All Hazards / Future relevance**
- `INFORM CC Risk Index` (2050, pessimistic)

**Hazard-specific / Future relevance (2050, pessimistic)**
- `Change in Hazard & Exposure` for **Drought**
- `Change in Hazard & Exposure` for **Flood** → mapped to your **Riverine Flooding Continued & Cluster** (unsplit flood proxy; flagged in notes)
- `Change in Hazard & Exposure` for **Coastal flood** → mapped to **Storm Surge** proxy

---
## Outputs
- `data/intermediate/inform_cc_extracted_2050_pessimistic_long.csv`
- `data/intermediate/inform_cc_matching_report.csv` (which names matched / unmatched)
- `data/intermediate/inform_cc_coverage_2050_pessimistic.csv`
- upserted master: `data/intermediate/wp3_country_indicators_long.csv`

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import unicodedata
import re
from difflib import get_close_matches

PROJECT_ROOT = Path(".").resolve()

# Inputs (repo-style)
COUNTRY_SCOPE_PATH = PROJECT_ROOT.parent / "data" / "raw" / "scope" / "ISO3_Regions.csv"
INFORM_RISK_XLSX   = PROJECT_ROOT.parent / "data" / "raw" / "INFORM" / "INFORM_Risk_Mid_2025_v071.xlsx"
INFORM_CC_XLSX     = PROJECT_ROOT.parent / "data" / "raw" / "INFORM" / "INFORM CC Brochure data.xlsx"

# Fallbacks for sandbox
FALLBACK_SCOPE = Path("/mnt/data/ISO3_Regions.csv")
FALLBACK_RISK  = Path("/mnt/data/INFORM_Risk_Mid_2025_v071.xlsx")
FALLBACK_CC    = Path("/mnt/data/INFORM CC Brochure data.xlsx")

if not COUNTRY_SCOPE_PATH.exists() and FALLBACK_SCOPE.exists():
    COUNTRY_SCOPE_PATH = FALLBACK_SCOPE
if not INFORM_RISK_XLSX.exists() and FALLBACK_RISK.exists():
    INFORM_RISK_XLSX = FALLBACK_RISK
if not INFORM_CC_XLSX.exists() and FALLBACK_CC.exists():
    INFORM_CC_XLSX = FALLBACK_CC

MASTER_PATH = PROJECT_ROOT.parent / "data" / "intermediate" / "wp3_country_indicators_long.csv"
MASTER_PATH.parent.mkdir(parents=True, exist_ok=True)

OUT_DIR = PROJECT_ROOT.parent / "data" / "intermediate"
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR = 2050
SCENARIO = "pessimistic"
SCENARIO_DETAIL = "RCP8.5+SSP3"

OUT_LONG = OUT_DIR / "inform_cc_extracted_2050_pessimistic_long.csv"
OUT_MATCH = OUT_DIR / "inform_cc_matching_report.csv"
OUT_COVERAGE = OUT_DIR / "inform_cc_coverage_2050_pessimistic.csv"

print("COUNTRY_SCOPE_PATH:", COUNTRY_SCOPE_PATH, "exists:", COUNTRY_SCOPE_PATH.exists())
print("INFORM_RISK_XLSX:", INFORM_RISK_XLSX, "exists:", INFORM_RISK_XLSX.exists())
print("INFORM_CC_XLSX:", INFORM_CC_XLSX, "exists:", INFORM_CC_XLSX.exists())
print("MASTER_PATH:", MASTER_PATH)

COUNTRY_SCOPE_PATH: C:\pipelines\sewa-multihazar\data\raw\scope\ISO3_Regions.csv exists: True
INFORM_RISK_XLSX: C:\pipelines\sewa-multihazar\data\raw\INFORM\INFORM_Risk_Mid_2025_v071.xlsx exists: True
INFORM_CC_XLSX: C:\pipelines\sewa-multihazar\data\raw\INFORM\INFORM CC Brochure data.xlsx exists: True
MASTER_PATH: C:\pipelines\sewa-multihazar\data\intermediate\wp3_country_indicators_long.csv


## Shared contract + upsert

In [2]:
HAZARDS_9 = [
    "Drought",
    "Flash Flooding",
    "Riverine Flooding Continued & Cluster",
    "Heatwave",
    "Storm Surge",
    "Wind",
    "Thunderstorm",
    "Wildfires",
    "Dust",
]
DIMENSIONS_5 = ["Prevalence", "Scale", "Impact", "Cascade impacts", "Future relevance"]

SOURCE = "INFORM Climate Change"

LONG_COLUMNS = [
    "iso3","country_name","region","hazard","dimension","source",
    "indicator_id","indicator_name","value_raw","unit_raw",
    "year","time_window","notes"
]
UPSERT_KEY = ["iso3","hazard","dimension","source","indicator_id","year"]

def assert_contract(df: pd.DataFrame) -> None:
    missing_cols = [c for c in LONG_COLUMNS if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Contract violation: missing columns {missing_cols}")
    bad_dims = sorted(set(df["dimension"]) - set(DIMENSIONS_5))
    if bad_dims:
        raise ValueError(f"Unknown dimensions {bad_dims}")
    allowed_haz = set(HAZARDS_9) | {"All Hazards"}
    bad_haz = sorted(set(df["hazard"]) - allowed_haz)
    if bad_haz:
        raise ValueError(f"Unknown hazards {bad_haz}")
    dup = df.duplicated(UPSERT_KEY, keep=False)
    if dup.any():
        raise ValueError("Duplicate UPSERT keys found. Check indicator_id or year assignment.")

def upsert_to_master(new_df: pd.DataFrame, master_path: Path) -> pd.DataFrame:
    assert_contract(new_df)
    if master_path.exists():
        master = pd.read_csv(master_path)
        for col in LONG_COLUMNS:
            if col not in master.columns:
                master[col] = np.nan
        master = master[LONG_COLUMNS]
        master_key = master[UPSERT_KEY].astype(str).agg("||".join, axis=1)
        new_key    = new_df[UPSERT_KEY].astype(str).agg("||".join, axis=1)
        master = master.loc[~master_key.isin(set(new_key))].copy()
        out = pd.concat([master, new_df[LONG_COLUMNS]], ignore_index=True)
    else:
        out = new_df[LONG_COLUMNS].copy()
    out.to_csv(master_path, index=False)
    return out

def norm_name(s: str) -> str:
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).strip().lower()
    # strip accents
    s = "".join(ch for ch in unicodedata.normalize("NFKD", s) if not unicodedata.combining(ch))
    # standardise punctuation/whitespace
    s = re.sub(r"[’'`]", "'", s)
    s = re.sub(r"[^a-z0-9\s'-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

## Step 1 — Load scope list

In [3]:
scope_raw = pd.read_csv(COUNTRY_SCOPE_PATH)
scope_raw.columns = [c.replace("\xa0"," ").strip() for c in scope_raw.columns]

scope = scope_raw.copy()
scope["Region"] = scope["Region"].replace({np.nan: None}).ffill()
scope["Region"] = scope["Region"].astype(str).str.replace("\xa0","", regex=False).str.strip()
scope["Country"] = scope["Country"].astype(str).str.replace("\xa0","", regex=False).str.strip()
scope["ISO3.Code"] = scope["ISO3.Code"].astype(str).str.strip().str.upper()

scope = scope.rename(columns={"Region":"region","Country":"country_name","ISO3.Code":"iso3"})
scope = scope[["iso3","country_name","region"]].drop_duplicates()

scope["country_norm"] = scope["country_name"].map(norm_name)
scope.head()

,iso3,country_name,region,country_norm
0,NGA,Nigeria,West Africa,nigeria
1,GHA,Ghana,West Africa,ghana
2,SEN,Senegal,West Africa,senegal
3,BFA,Burkina Faso,West Africa,burkina faso
4,MLI,Mali,West Africa,mali


## Step 2 — Build ISO3 dictionary from INFORM Risk (country + ISO3)

In [4]:
SHEET = "INFORM Risk Mid 2025 (a-z)"
raw = pd.read_excel(INFORM_RISK_XLSX, sheet_name=SHEET, header=None)
headers = raw.iloc[1].tolist()
df = raw.iloc[3:].copy()
df.columns = headers
df = df.rename(columns={"COUNTRY":"country_inform","ISO3":"iso3"})
df["iso3"] = df["iso3"].astype(str).str.strip().str.upper()
df["country_inform"] = df["country_inform"].astype(str).str.strip()

inform_names = df[["iso3","country_inform"]].drop_duplicates()
inform_names["country_norm"] = inform_names["country_inform"].map(norm_name)

# Manual alias layer for known variants (add here as you encounter edge cases)
# Keys and values are normalized (norm_name)
ALIASES = {
    norm_name("Cote d'Ivoire"): norm_name("Côte d'Ivoire"),
    norm_name("Sao Tome and Principe"): norm_name("Sao Tome and Principe"),
    norm_name("São Tomé and Príncipe"): norm_name("Sao Tome and Principe"),
    norm_name("Congo (Brazzaville)"): norm_name("Congo"),
    norm_name("Congo, Rep."): norm_name("Congo"),
    norm_name("Republic of the Congo"): norm_name("Congo"),
    norm_name("DR Congo"): norm_name("Congo DR"),
    norm_name("Congo, Dem. Rep."): norm_name("Congo DR"),
    norm_name("Democratic Republic of the Congo"): norm_name("Congo DR"),
    norm_name("Eswatini (Swaziland)"): norm_name("Eswatini"),
    norm_name("Swaziland"): norm_name("Eswatini"),
    norm_name("Gambia"): norm_name("Gambia"),
    norm_name("Guinea-Bissau"): norm_name("Guinea Bissau"),
}

# Build dict: normalized name -> iso3
name_to_iso3 = dict(zip(inform_names["country_norm"], inform_names["iso3"]))

# Also add scope names (if INFORM uses different formatting) via alias mapping
for _, r in scope.iterrows():
    cn = r["country_norm"]
    if cn in ALIASES:
        cn = ALIASES[cn]
    if cn in name_to_iso3:
        continue

print("ISO3 dictionary size:", len(name_to_iso3))

ISO3 dictionary size: 191


c:\Users\duruenaramirez\AppData\Local\miniforge3\envs\ibf-env\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Unknown extension is not supported and will be removed
  for idx, row in parser.parse():
c:\Users\duruenaramirez\AppData\Local\miniforge3\envs\ibf-env\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


## Step 3 — Read INFORM CC brochure data and extract the pessimistic 2050 columns

In [5]:
cc_raw = pd.read_excel(INFORM_CC_XLSX, header=None)

# The sheet uses multi-row headers:
# - row 0: section headers (baseline / mid-century / end-century)
# - row 2: scenario labels (e.g., "PESSIMISTIC (P) climate..." for mid-century)
# - row 4: column names
# - data begins at row 5

row0 = cc_raw.iloc[0].tolist()
row2 = cc_raw.iloc[2].tolist()
row4 = cc_raw.iloc[4].tolist()

data = cc_raw.iloc[5:].copy()
data.columns = [f"c{i}" for i in range(cc_raw.shape[1])]

# Identify the key columns by their known positions (as per official brochure layout)
# Mid-century (≈2050) risk index — pessimistic scenario is column 2 in the brochure table.
COL_CC_RISK_2050_P = "c2"

# Change in Hazard & Exposure for individual hazards (2050):
# Flood (P)  -> column 14
# Coastal flood (P) -> column 17
# Drought (P) -> column 20
COL_CHANGE_FLOOD_P   = "c14"
COL_CHANGE_COASTAL_P = "c17"
COL_CHANGE_DROUGHT_P = "c20"

# Country name column is c0
COL_COUNTRY = "c0"

cc = data[[COL_COUNTRY, COL_CC_RISK_2050_P, COL_CHANGE_FLOOD_P, COL_CHANGE_COASTAL_P, COL_CHANGE_DROUGHT_P]].copy()
cc = cc.rename(columns={
    COL_COUNTRY: "country_cc",
    COL_CC_RISK_2050_P: "cc_risk_2050_p",
    COL_CHANGE_FLOOD_P: "chg_flood_2050_p",
    COL_CHANGE_COASTAL_P: "chg_coastal_2050_p",
    COL_CHANGE_DROUGHT_P: "chg_drought_2050_p",
})

# Numeric conversion
for c in ["cc_risk_2050_p","chg_flood_2050_p","chg_coastal_2050_p","chg_drought_2050_p"]:
    cc[c] = pd.to_numeric(cc[c], errors="coerce")

cc["country_cc"] = cc["country_cc"].astype(str).str.strip()
cc["country_norm"] = cc["country_cc"].map(norm_name)

cc.head(10)

,country_cc,cc_risk_2050_p,chg_flood_2050_p,chg_coastal_2050_p,chg_drought_2050_p,country_norm
5,Afghanistan,8.1,-1.3,0.0,2.8,afghanistan
6,Albania,2.7,-0.9,0.0,2.7,albania
7,Algeria,4.1,0.3,2.7,2.8,algeria
8,Angola,5.4,1.2,1.1,2.1,angola
9,Antigua and Barbuda,2.2,5.2,0.0,0.0,antigua and barbuda
10,Argentina,3.2,0.8,0.6,1.6,argentina
11,Armenia,5.4,-1.0,0.0,3.2,armenia
12,Australia,2.6,0.9,1.1,2.1,australia
13,Austria,2.0,-0.1,0.0,1.7,austria
14,Azerbaijan,5.9,0.5,0.0,3.0,azerbaijan


## Step 4 — Match INFORM CC countries to ISO3 (via INFORM Risk dictionary), with a matching report

In [6]:
def map_to_iso3(country_norm: str) -> tuple[str, str]:
    """Return (iso3, match_type). match_type documents how we mapped."""
    if not country_norm:
        return ("", "empty")

    # Apply aliases first
    key = ALIASES.get(country_norm, country_norm)

    if key in name_to_iso3:
        return (name_to_iso3[key], "direct_or_alias")

    # Fuzzy fallback: close match against INFORM norm names
    candidates = list(name_to_iso3.keys())
    matches = get_close_matches(key, candidates, n=1, cutoff=0.90)
    if matches:
        return (name_to_iso3[matches[0]], f"fuzzy:{matches[0]}")
    return ("", "unmatched")

mapped = cc.copy()
mapped[["iso3","match_type"]] = mapped["country_norm"].apply(lambda x: pd.Series(map_to_iso3(x)))

# Join to scope (filter to WP3 countries)
mapped = mapped.merge(scope[["iso3","country_name","region"]], on="iso3", how="right")

# Matching report (for manual review)
report = mapped[["iso3","country_name","region","country_cc","match_type"]].drop_duplicates()
report.to_csv(OUT_MATCH, index=False)

print("Unmatched scope countries (no CC data or mapping):")
unmatched_scope = report.loc[report["country_cc"].isna(), ["iso3","country_name","region"]]
print(unmatched_scope["iso3"].tolist())

print("Rows with CC present:", report["country_cc"].notna().sum(), "of", len(report))
report.head(20)

Unmatched scope countries (no CC data or mapping):
['REU']
Rows with CC present: 46 of 47


,iso3,country_name,region,country_cc,match_type
0,NGA,Nigeria,West Africa,Nigeria,direct_or_alias
1,GHA,Ghana,West Africa,Ghana,direct_or_alias
2,SEN,Senegal,West Africa,Senegal,direct_or_alias
3,BFA,Burkina Faso,West Africa,Burkina Faso,direct_or_alias
4,MLI,Mali,West Africa,Mali,direct_or_alias
5,NER,Niger,West Africa,Niger,direct_or_alias
6,CIV,Côte d'Ivoire,West Africa,Côte d'Ivoire,direct_or_alias
7,BEN,Benin,West Africa,Benin,direct_or_alias
8,TGO,Togo,West Africa,Togo,direct_or_alias
9,SLE,Sierra Leone,West Africa,Sierra Leone,direct_or_alias


## Step 5 — Build long-format indicators and upsert to master

In [7]:
# Build long table (only where values exist)
rows = []

# All hazards / Future relevance: CC risk index (2050, pessimistic)
tmp = mapped[["iso3","country_name","region","cc_risk_2050_p"]].copy()
tmp = tmp.rename(columns={"cc_risk_2050_p":"value_raw"})
tmp["hazard"] = "All Hazards"
tmp["dimension"] = "Future relevance"
tmp["source"] = SOURCE
tmp["indicator_id"] = f"INFORMCC.RISK_INDEX.2050.{SCENARIO}"
tmp["indicator_name"] = "INFORM CC Risk Index (2050, pessimistic)"
tmp["unit_raw"] = "index_0_10"
tmp["year"] = YEAR
tmp["time_window"] = "scenario_snapshot"
tmp["notes"] = f"Scenario={SCENARIO} ({SCENARIO_DETAIL}). Mid-century (≈2050)."
rows.append(tmp)

# Hazard-specific change in Hazard & Exposure (2050, pessimistic)
def add_change(col, hazard, indicator_id, indicator_name, notes_extra=""):
    t = mapped[["iso3","country_name","region", col]].copy()
    t = t.rename(columns={col:"value_raw"})
    t["hazard"] = hazard
    t["dimension"] = "Future relevance"
    t["source"] = SOURCE
    t["indicator_id"] = indicator_id
    t["indicator_name"] = indicator_name
    t["unit_raw"] = "delta_index_0_10"
    t["year"] = YEAR
    t["time_window"] = "scenario_snapshot"
    t["notes"] = f"Scenario={SCENARIO} ({SCENARIO_DETAIL}). Change in Hazard & Exposure vs baseline. {notes_extra}".strip()
    return t

# Flood is unsplit in INFORM CC -> mapped to Riverine Flooding Continued & Cluster with explicit note
rows.append(add_change("chg_flood_2050_p",
                       "Riverine Flooding Continued & Cluster",
                       f"INFORMCC.CHG_HAZEX.FLOOD.2050.{SCENARIO}",
                       "Change in Hazard & Exposure: Flood (2050, pessimistic) — mapped to Riverine Flooding Continued & Cluster",
                       notes_extra="INFORM CC provides an unsplit 'Flood' hazard; mapped to Riverine Flooding category for WP3 with this caveat."))

rows.append(add_change("chg_coastal_2050_p",
                       "Storm Surge",
                       f"INFORMCC.CHG_HAZEX.COASTAL_FLOOD.2050.{SCENARIO}",
                       "Change in Hazard & Exposure: Coastal flood (2050, pessimistic) — proxy for Storm Surge",
                       notes_extra="INFORM CC uses 'Coastal flood'; treated as proxy for Storm Surge in WP3 taxonomy."))

rows.append(add_change("chg_drought_2050_p",
                       "Drought",
                       f"INFORMCC.CHG_HAZEX.DROUGHT.2050.{SCENARIO}",
                       "Change in Hazard & Exposure: Drought (2050, pessimistic)"))

long_df = pd.concat(rows, ignore_index=True)

# Drop missing values
long_df = long_df.loc[~long_df["value_raw"].isna()].copy()
long_df = long_df[LONG_COLUMNS]

assert_contract(long_df)

# Write dataset extract
long_df.to_csv(OUT_LONG, index=False)

# Coverage
cov_rows = []
for haz, col in [
    ("All Hazards", "cc_risk_2050_p"),
    ("Drought", "chg_drought_2050_p"),
    ("Riverine Flooding Continued & Cluster", "chg_flood_2050_p"),
    ("Storm Surge", "chg_coastal_2050_p"),
]:
    t = mapped[["iso3","country_name","region", col]].copy()
    t["hazard"] = haz
    t["source"] = SOURCE
    t["present"] = ~t[col].isna()
    t["notes"] = f"Presence derived from INFORM CC column '{col}' not-missing (scenario={SCENARIO_DETAIL})."
    cov_rows.append(t[["iso3","country_name","region","hazard","source","present","notes"]])

coverage = pd.concat(cov_rows, ignore_index=True)
coverage.to_csv(OUT_COVERAGE, index=False)

# Upsert
master = upsert_to_master(long_df, MASTER_PATH)

print("Wrote:", OUT_LONG)
print("Wrote:", OUT_MATCH)
print("Wrote:", OUT_COVERAGE)
print("Updated master:", MASTER_PATH, "rows:", len(master))

long_df.head(12)

Wrote: C:\pipelines\sewa-multihazar\data\intermediate\inform_cc_extracted_2050_pessimistic_long.csv
Wrote: C:\pipelines\sewa-multihazar\data\intermediate\inform_cc_matching_report.csv
Wrote: C:\pipelines\sewa-multihazar\data\intermediate\inform_cc_coverage_2050_pessimistic.csv
Updated master: C:\pipelines\sewa-multihazar\data\intermediate\wp3_country_indicators_long.csv rows: 736


,iso3,country_name,region,hazard,dimension,source,indicator_id,indicator_name,value_raw,unit_raw,year,time_window,notes
0,NGA,Nigeria,West Africa,All Hazards,Future relevance,INFORM Climate Change,INFORMCC.RISK_INDEX.2050.pessimistic,"INFORM CC Risk Index (2050, pessimistic)",6.7,index_0_10,2050,scenario_snapshot,Scenario=pessimistic (RCP8.5+SSP3). Mid-centur...
1,GHA,Ghana,West Africa,All Hazards,Future relevance,INFORM Climate Change,INFORMCC.RISK_INDEX.2050.pessimistic,"INFORM CC Risk Index (2050, pessimistic)",4.4,index_0_10,2050,scenario_snapshot,Scenario=pessimistic (RCP8.5+SSP3). Mid-centur...
2,SEN,Senegal,West Africa,All Hazards,Future relevance,INFORM Climate Change,INFORMCC.RISK_INDEX.2050.pessimistic,"INFORM CC Risk Index (2050, pessimistic)",5.2,index_0_10,2050,scenario_snapshot,Scenario=pessimistic (RCP8.5+SSP3). Mid-centur...
3,BFA,Burkina Faso,West Africa,All Hazards,Future relevance,INFORM Climate Change,INFORMCC.RISK_INDEX.2050.pessimistic,"INFORM CC Risk Index (2050, pessimistic)",6.6,index_0_10,2050,scenario_snapshot,Scenario=pessimistic (RCP8.5+SSP3). Mid-centur...
4,MLI,Mali,West Africa,All Hazards,Future relevance,INFORM Climate Change,INFORMCC.RISK_INDEX.2050.pessimistic,"INFORM CC Risk Index (2050, pessimistic)",7.1,index_0_10,2050,scenario_snapshot,Scenario=pessimistic (RCP8.5+SSP3). Mid-centur...
5,NER,Niger,West Africa,All Hazards,Future relevance,INFORM Climate Change,INFORMCC.RISK_INDEX.2050.pessimistic,"INFORM CC Risk Index (2050, pessimistic)",7.5,index_0_10,2050,scenario_snapshot,Scenario=pessimistic (RCP8.5+SSP3). Mid-centur...
6,CIV,Côte d'Ivoire,West Africa,All Hazards,Future relevance,INFORM Climate Change,INFORMCC.RISK_INDEX.2050.pessimistic,"INFORM CC Risk Index (2050, pessimistic)",5.2,index_0_10,2050,scenario_snapshot,Scenario=pessimistic (RCP8.5+SSP3). Mid-centur...
7,BEN,Benin,West Africa,All Hazards,Future relevance,INFORM Climate Change,INFORMCC.RISK_INDEX.2050.pessimistic,"INFORM CC Risk Index (2050, pessimistic)",4.9,index_0_10,2050,scenario_snapshot,Scenario=pessimistic (RCP8.5+SSP3). Mid-centur...
8,TGO,Togo,West Africa,All Hazards,Future relevance,INFORM Climate Change,INFORMCC.RISK_INDEX.2050.pessimistic,"INFORM CC Risk Index (2050, pessimistic)",4.8,index_0_10,2050,scenario_snapshot,Scenario=pessimistic (RCP8.5+SSP3). Mid-centur...
9,SLE,Sierra Leone,West Africa,All Hazards,Future relevance,INFORM Climate Change,INFORMCC.RISK_INDEX.2050.pessimistic,"INFORM CC Risk Index (2050, pessimistic)",5.3,index_0_10,2050,scenario_snapshot,Scenario=pessimistic (RCP8.5+SSP3). Mid-centur...
